# Summary: predicting next-month default

**Data.** 30,000 credit card customers (UCI Default of Credit Card Clients, Taiwan, 2005). 22.1% missed the following month's payment. The data was split 80/20 (stratified, `random_state=42`) into 24,000 training and 6,000 test customers; the test set contains 1,327 defaulters.

**Inputs.** Repayment status for the last six months (`pay_1` to `pay_6`), credit limit, and credit utilisation (latest bill divided by limit). Sex, marital status, age and education were deliberately excluded: sex, marriage and age are protected characteristics under UK equality law, and education can act as a proxy for them.

**Model.** Logistic regression on standardised inputs, chosen because its weights can be read directly.

## Results on the 6,000 held-out customers

| Model | AUC | Riskiest 10% flagged: share of defaulters caught | Riskiest 20% | Riskiest 30% |
|---|---|---|---|---|
| Lateness only (`pay_1` as a score) | 0.676 | n/a | n/a | n/a |
| Model 1: repayment codes as plain numbers | 0.698 | 31.1% | 47.6% | 56.2% |
| Model 2: repayment codes as categories | 0.745 | 30.5% | 48.6% | 60.0% |

- The repayment codes (-2, -1, 0, 1, 2, ...) are labels, not a numeric scale. Treating them as categories (one on/off switch per code; codes of 4 and above grouped) raised AUC from 0.698 to 0.745. The gain was mainly further down the ranking; in the riskiest 10% the two models performed about the same.
- Model 2's average predicted risk (0.221) matched the actual default rate (0.221). This is a basic check, not a full calibration test.
- The model is not clearly better than a simple lateness rule. In the SQL analysis, flagging customers 2+ months late (10.4% of customers) caught 32.8% of defaults, against 30.5% for the model's riskiest 10%. The model's advantage is that it gives every customer a probability and ranks better further down the list.

## What drives the predictions

Weights are on standardised inputs and show association, not cause. Compare their sizes with each other; they are not percentages.

| Direction | Clue | Weight |
|---|---|---|
| Raises predicted risk | 2 months late in the latest month | +0.47 |
| Lowers predicted risk | Paid the minimum in the latest month | -0.34 |
| Lowers predicted risk | Higher credit limit | -0.23 |
| Raises predicted risk | Higher credit utilisation | +0.11 |

This agrees with the SQL exploration: recent lateness is the strongest signal, and credit limit and utilisation are smaller ones. Rare or overlapping codes (for example 3 months late) have unstable weights and should not be read individually.

## Trade-off for a review team

| Flag riskiest | Customers flagged | Defaulters caught | Share of all defaulters caught | Share of flagged who default | Risk cutoff |
|---|---|---|---|---|---|
| 10% | 600 | 405 | 30.5% | 67.5% | 0.58 |
| 20% | 1,200 | 645 | 48.6% | 53.8% | 0.27 |
| 30% | 1,800 | 796 | 60.0% | 44.2% | 0.19 |
| 40% | 2,400 | 899 | 67.7% | 37.5% | 0.16 |

Moving from 20% to 30% adds 600 flagged customers, of whom about 25% default. Moving from 30% to 40% adds customers who default at about 17%, below the 22.1% average, so the extra flags are less risky than a typical customer.

## Recommendation

On this data, a review capacity of roughly 20% of customers (risk cutoff about 0.27) balances coverage and accuracy: it reaches about half of defaulters, and about half of flagged customers do default. Extending to 30% reaches 60% of defaulters with diminishing returns, and beyond 30% the extra flags are less risky than average. The right cutoff depends on the bank's cost of review relative to its loss on a default, which this dataset does not contain.

## Limitations

- Taiwan, 2005 data; not representative of UK lending today.
- "Default" is the source label (a missed payment the following month), not a regulatory definition.
- The repayment codes 0 and -2 are not documented in the source; they are commonly read as "paid the minimum" and "no consumption".
- One train/test split, with no cross-validation or confidence intervals. With 1,327 test defaulters, the percentages above carry sampling noise.
- Only the latest month's utilisation is used, and logistic regression cannot capture non-linear effects.
- Demographic variables were excluded, but other inputs may still correlate with them. No fairness testing was done.
- Weights show association, not cause.

In [3]:
import os
import pandas as pd

# Run from the folder that contains credit_default.csv
if not os.path.exists('credit_default.csv'):
    os.chdir('credit-risk-default-analysis')

df = pd.read_csv('credit_default.csv')
print(df.shape)
df.head()

(30000, 25)


,id,limit_bal,sex,education,marriage,age,pay_1,pay_2,pay_3,pay_4,...,bill_amt4,bill_amt5,bill_amt6,pay_amt1,pay_amt2,pay_amt3,pay_amt4,pay_amt5,pay_amt6,default_next_month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [4]:
print(df['default_next_month'].mean().round(3))
print(df.isna().sum().sum())
df['default_next_month'].value_counts()

0.221
0


default_next_month
0    23364
1     6636
Name: count, dtype: int64

In [5]:
df['utilisation'] = df['bill_amt1'] / df['limit_bal']

features = ['pay_1', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6',
            'limit_bal', 'utilisation']

X = df[features]
y = df['default_next_month']

print(X.shape, y.shape)
X.head()

(30000, 8) (30000,)


,pay_1,pay_2,pay_3,pay_4,pay_5,pay_6,limit_bal,utilisation
0,2,2,-1,-1,-2,-2,20000,0.195650
1,-1,2,0,0,0,2,120000,0.022350
2,0,0,0,0,0,0,90000,0.324878
3,0,0,0,0,0,0,50000,0.939800
4,-1,0,-1,0,0,0,50000,0.172340


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(24000, 8) (6000, 8)
0.221 0.221


In [9]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(X_train, y_train)

probs = model.predict_proba(X_test)[:, 1]
print(probs[:5].round(3))
print(y_test.values[:5])

[0.143 0.102 0.197 0.124 0.027]
[0 0 0 1 0]


In [10]:
from sklearn.metrics import roc_auc_score

print('Model AUC:', round(roc_auc_score(y_test, probs), 3))
print('Lateness-only AUC:', round(roc_auc_score(y_test, X_test['pay_1']), 3))

test = pd.DataFrame({'prob': probs, 'default': y_test.values})
test = test.sort_values('prob', ascending=False)
total_defaults = test['default'].sum()

for share in [0.10, 0.20, 0.30]:
    n = int(len(test) * share)
    caught = test['default'].head(n).sum() / total_defaults
    print(f'Flag riskiest {share:.0%} -> catches {caught:.1%} of defaults')

Model AUC: 0.698
Lateness-only AUC: 0.676
Flag riskiest 10% -> catches 31.1% of defaults
Flag riskiest 20% -> catches 47.6% of defaults
Flag riskiest 30% -> catches 56.2% of defaults


In [11]:
pay_cols = ['pay_1', 'pay_2', 'pay_3', 'pay_4', 'pay_5', 'pay_6']
pay_cats = pd.get_dummies(df[pay_cols].clip(upper=4).astype(str), prefix=pay_cols)

X2 = pd.concat([pay_cats, df[['limit_bal', 'utilisation']]], axis=1).astype(float)
print(X2.shape)

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.2, random_state=42, stratify=y
)

model2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
model2.fit(X2_train, y2_train)
probs2 = model2.predict_proba(X2_test)[:, 1]

print('Model 2 AUC:', round(roc_auc_score(y2_test, probs2), 3))

test2 = pd.DataFrame({'prob': probs2, 'default': y2_test.values})
test2 = test2.sort_values('prob', ascending=False)
for share in [0.10, 0.20, 0.30]:
    n = int(len(test2) * share)
    caught = test2['default'].head(n).sum() / test2['default'].sum()
    print(f'Flag riskiest {share:.0%} -> catches {caught:.1%} of defaults')

(30000, 42)
Model 2 AUC: 0.745
Flag riskiest 10% -> catches 30.5% of defaults
Flag riskiest 20% -> catches 48.6% of defaults
Flag riskiest 30% -> catches 60.0% of defaults


In [12]:
weights = pd.Series(model2[-1].coef_[0], index=X2.columns)

print('Clues that push risk UP:')
print(weights.sort_values(ascending=False).head(8).round(2))

print()
print('Clues that push risk DOWN:')
print(weights.sort_values().head(5).round(2))

Clues that push risk UP:
pay_1_2        0.47
pay_1_3        0.17
pay_1_1        0.15
pay_3_2        0.11
utilisation    0.11
pay_4_2        0.09
pay_1_4        0.07
pay_6_4        0.07
dtype: float64

Clues that push risk DOWN:
pay_1_0     -0.34
limit_bal   -0.23
pay_1_-2    -0.16
pay_6_0     -0.09
pay_2_-1    -0.07
dtype: float64


In [13]:
total_defaults = test2['default'].sum()
rows = []

for share in [0.10, 0.20, 0.30, 0.40]:
    n = int(len(test2) * share)
    top = test2.head(n)
    caught = top['default'].sum()
    rows.append({
        'flag_top': f'{share:.0%}',
        'customers_flagged': n,
        'defaulters_caught': caught,
        'pct_of_defaulters_caught': round(100 * caught / total_defaults, 1),
        'pct_flagged_who_default': round(100 * caught / n, 1),
        'flagged_but_paid': n - caught,
        'risk_cutoff': round(top['prob'].min(), 2),
    })

print('Average predicted risk:', round(probs2.mean(), 3),
      '| actual default rate:', round(y2_test.mean(), 3))
pd.DataFrame(rows)

Average predicted risk: 0.221 | actual default rate: 0.221


,flag_top,customers_flagged,defaulters_caught,pct_of_defaulters_caught,pct_flagged_who_default,flagged_but_paid,risk_cutoff
0,10%,600,405,30.5,67.5,195,0.58
1,20%,1200,645,48.6,53.8,555,0.27
2,30%,1800,796,60.0,44.2,1004,0.19
3,40%,2400,899,67.7,37.5,1501,0.16
